In [ ]:
from sqlalchemy import create_engine
from project_starter import init_database

db_engine = create_engine("sqlite:///munder_difflin.db")
init_database(db_engine)

In [ ]:
import pandas as pd
import sqlite3

# Connect to your SQLite database
conn = sqlite3.connect("munder_difflin.db")

# Query the sqlite_master system table
query = "SELECT name FROM sqlite_master WHERE type='table';"
df_tables = pd.read_sql_query(query, conn)
df_tables

In [ ]:
pd.read_sql('select * from transactions', conn)

In [ ]:
pd.read_sql('select * from quote_requests', conn)

In [ ]:
pd.read_sql('select * from quotes', conn)

In [ ]:
pd.read_sql('select * from inventory', conn)

In [ ]:
import pandas as pd
from project_starter import paper_supplies

paper_supplies_df = pd.DataFrame.from_dict(paper_supplies)
quote_requests_df = pd.read_csv('quote_requests.csv')

### Configuration

In [1]:
import os
import enum
from dotenv import load_dotenv
from smolagents import OpenAIServerModel, ToolCallingAgent, tool
from project_starter import get_stock_level
from project_starter import get_cash_balance
from project_starter import get_all_inventory
from project_starter import create_transaction
from project_starter import search_quote_history
from project_starter import get_supplier_delivery_date

load_dotenv()

model = OpenAIServerModel(
    model_id='gpt-5-nano',
    api_base=os.getenv('OPENAI_BASE_URL'),
    api_key=os.getenv('OPENAI_API_KEY'),
)

get_all_inventory = tool(get_all_inventory)
get_stock_level = tool(get_stock_level)
get_supplier_delivery_date = tool(get_supplier_delivery_date)
get_cash_balance = tool(get_cash_balance)
create_transaction = tool(create_transaction)

/Users/jameschang/Downloads/agentic-ai-c4-exercises-demos/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### RequestUnderstandingAgent

In [10]:
import csv
import json
from enum import Enum
from datetime import date
from typing import Literal
from smolagents import ToolCallingAgent
from pydantic import BaseModel, ConfigDict, Field, model_validator
from project_starter import paper_supplies


catalog = {
    item['item_name']: item
    for item in paper_supplies
}

with open('quote_requests.csv', encoding='utf-8', newline="") as file:
    historical_requests = list(csv.DictReader(file))

ItemName = Enum(
    'ItemName',
    {
        f'ITEM_{i}': item_name
        for i, item_name in enumerate(catalog)
    }
)
ItemCategory = Enum(
    'ItemCategory', 
    {f'CATEGOR_{i}': category
    for i, category in enumerate(
        sorted({item['category'] for item in paper_supplies})
    )},
    type=str
)


class RequestedItem(BaseModel):

    model_config = ConfigDict(extra='forbid')
    customer_description: str
    match_status: Literal['matched', 'not_offered', 'needs_clarification']
    item_name: ItemName | None
    category: ItemCategory | None
    units: int | None = Field(strict=True, gt=0)
    clarification_question: str | None

    @model_validator(mode='after')
    def check_match(self):
        if self.match_status == 'matched':
            if self.item_name is None or self.category is None:
                raise ValueError('A matched item requires item_name and category.')
            expected_category = catalog[self.item_name.value]['category']
            if self.category.value != expected_category:
                raise ValueError('Item name and category do not match')
        else:
            if self.item_name is not None or self.category is not None:
                raise ValueError('item_name and category must be None unless the item is matched.')
        if (
            self.match_status == 'needs_clarification'
            or (
                self.match_status == 'matched'
                and self.units is None
            )
        ) and not self.clarification_question:
            raise ValueError('A clarification question is required')
        return self


class ParsedItems(BaseModel):
    model_config = ConfigDict(extra='forbid')
    items: list[RequestedItem] = Field(min_length=1)
    delivery_by: date | None = Field(
        description=(
            "The customer's latest acceptable delivery date, "
            "in YYYY-MM-DD format. Use null if missing or ambiguous."
        )
    )
    delivery_date_question: str | None = Field(
        description=(
            "A clarification question if the delivery deadline "
            "is missing or ambiguous; otherwise null."
        )
    )

    @model_validator(mode='after')
    def check_delivery_date(self):
        if self.delivery_by is None and not self.delivery_date_question:
            raise ValueError("A delivery date clarification question is required.")
        return self


schema_text = json.dumps(
    ParsedItems.model_json_schema(),
    ensure_ascii=False,
    indent=2
)

catalog_text = json.dumps(
    {
        name: item['category']
        for name, item in catalog.items()
    },
    ensure_ascii=False,
    indent=2
)

class RequestUnderstandingAgent(ToolCallingAgent):
    def __init__(self, model: OpenAIServerModel):
        super().__init__(
            tools=[],
            model=model,
            name='request_understanding_agent',
            description=(
                "Parses customer requests, matches catalog products, "
                "and identifies missing information."
            ),
            instructions=(
                "You are a customer request understanding agent.\n"
                "Convert the customer's message into the required structured output.\n"
                "Do not check inventory, calculate prices, execute transactions, "
                "or promise delivery.\n"
                "\n"
                "PRODUCT MATCHING\n"
                "- Match customer descriptions to catalog products by meaning, "
                "not exact wording. Clear synonyms and paraphrases are allowed.\n"
                "- Use only item_name values from the catalog.\n"
                "- For matched items, copy the corresponding category from "
                "the catalog mapping. Do not classify category independently.\n"
                "- Preserve the customer's explicit size, color, material, "
                "weight, and other specifications.\n"
                "- Do not invent specifications or silently substitute products.\n"
                "- Do not assume a catalog item supports a specification "
                "that the catalog does not establish.\n"
                "- Copy customer_description verbatim from the relevant "
                "part of the customer's message.\n"
                "\n"
                "MATCH STATUS AND FIELD CONSISTENCY\n"
                "- Use matched when the requested product can be reliably "
                "mapped to a catalog item.\n"
                "- When matched, item_name and category must both be non-null "
                "and must form an exact pair from the catalog mapping.\n"
                "- Use not_offered when the requested product is clearly "
                "outside the catalog.\n"
                "- Use needs_clarification when the product is ambiguous, "
                "multiple matches remain possible, or a required specification "
                "cannot be established from the catalog.\n"
                "- When not_offered or needs_clarification, item_name and "
                "category MUST BOTH be null. Never include a tentative match "
                "in these fields.\n"
                "- When needs_clarification, provide a specific, non-empty "
                "clarification_question.\n"
                "- Missing quantity alone does not change a reliable product "
                "match into needs_clarification; keep match_status as matched "
                "and ask for the quantity.\n"
                "\n"
                "QUANTITIES\n"
                "- Extract units as a positive integer when the quantity "
                "and applicable selling unit are clear.\n"
                "- If quantity is missing, ambiguous, or requires an unknown "
                "package conversion, set units to null and ask for clarification.\n"
                "- Never guess the number of sheets or individual items "
                "inside a pack, box, or other package.\n"
                "- For an unmatched product, you may preserve an explicitly "
                "stated positive integer quantity without implying a catalog match.\n"
                "- Set clarification_question to null when no item-level "
                "clarification is needed.\n"
                "\n"
                "DELIVERY DEADLINE\n"
                "- Extract the latest acceptable delivery date as delivery_by "
                "in YYYY-MM-DD format.\n"
                "- Preserve explicitly stated historical dates, including their year.\n"
                "- Resolve relative dates only when an explicitly supplied "
                "request date provides sufficient context.\n"
                "- Never use the computer's current date as an assumed request date.\n"
                "- If the deadline is missing or ambiguous, set delivery_by "
                "to null and provide a non-empty delivery_date_question.\n"
                "- If different items have different deadlines, set delivery_by "
                "to null and ask for clarification; this schema supports "
                "one shared deadline.\n"
                "- If the shared deadline is clear, set delivery_date_question "
                "to null.\n"
                "- Extract the requested deadline without deciding whether "
                "it can be met.\n"
                "\n"
                "OUTPUT\n"
                "- Return a JSON object as your final answer, with no Markdown "
                "fences or surrounding commentary.\n"
                "- Include all required fields and no extra fields.\n"
                "- Return every requested item, including unsupported items "
                "and items requiring clarification.\n"
                "- Put follow-up questions in the designated fields for the "
                "Orchestrator to present to the customer.\n"
                "- Before returning, check that match_status, item_name, "
                "category, units, and clarification_question are consistent.\n"
                "\n"
                "CATALOG ITEM-TO-CATEGORY MAPPING\n"
                f"{catalog_text}\n"
                "\n"
                "REQUIRED OUTPUT JSON SCHEMA\n"
                f"{schema_text}"
            )
        )

request_understanding_agent = RequestUnderstandingAgent(model=model)
result = request_understanding_agent.run("""I would like to request the following paper supplies for the ceremony: 

- 200 sheets of A4 glossy paper
- 100 sheets of heavy cardstock (white)
- 100 sheets of colored paper (assorted colors)

I need these supplies delivered by April 15, 2025. Thank you.""")

parsed = (
    ParsedItems.model_validate_json(result)
    if isinstance(result, str)
    else ParsedItems.model_validate(result)
)

print(parsed.model_dump_json(indent=2))

╭───────────────────────────────────── New run - request_understanding_agent ─────────────────────────────────────╮
│                                                                                                                 │
│ I would like to request the following paper supplies for the ceremony:                                          │
│                                                                                                                 │
│ - 200 sheets of A4 glossy paper                                                                                 │
│ - 100 sheets of heavy cardstock (white)                                                                         │
│ - 100 sheets of colored paper (assorted colors)                                                                 │
│                                                                                                                 │
│ I need these supplies delivered by April 15, 2025. Thank you.                                                   │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5-nano ──────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': '{"items":[{"customer_description":"200 sheets of A4    │
│ glossy                                                                                                          │
│ paper","match_status":"needs_clarification","item_name":null,"category":null,"units":200,"clarification_questio │
│ n":"Please confirm which catalog item to use for \'A4 glossy paper\': \'A4 paper\' or \'Glossy paper\'. We do   │
│ not have a single catalog item that combines size and finish."},{"customer_description":"100 sheets of heavy    │
│ cardstock                                                                                                       │
│ (white)","match_status":"needs_clarification","item_name":null,"category":null,"units":100,"clarification_quest │
│ ion":"Please choose the exact catalog item for heavy cardstock: \'Cardstock\' or \'250 gsm cardstock\'. Also    │
│ confirm color (white) if required by the catalog item."},{"customer_description":"100 sheets of colored paper   │
│ (assorted colors)","match_status":"matched","item_name":"Colored                                                │
│ paper","category":"paper","units":100,"clarification_question":null}],"delivery_by":"2025-04-15","delivery_date │
│ _question":null}'}                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"items":|{"customer_description":"200 sheets of A4 glossy 
paper","match_status":"needs_clarification","item_name":null,"category":null,"units":200,"clarification_question":"
Please confirm which catalog item to use for 'A4 glossy paper': 'A4 paper' or 'Glossy paper'. We do not have a 
single catalog item that combines size and finish."},{"customer_description":"100 sheets of heavy cardstock 
(white)","match_status":"needs_clarification","item_name":null,"category":null,"units":100,"clarification_question"
:"Please choose the exact catalog item for heavy cardstock: 'Cardstock' or '250 gsm cardstock'. Also confirm color 
(white) if required by the catalog item."},{"customer_description":"100 sheets of colored paper (assorted 
colors)","match_status":"matched","item_name":"Colored 
paper","category":"paper","units":100,"clarification_question":null}],"delivery_by":"2025-04-15","delivery_date_que
stion":null}

Final answer: {"items":[{"customer_description":"200 sheets of A4 glossy 
paper","match_status":"needs_clarification","item_name":null,"category":null,"units":200,"clarification_question":"
Please confirm which catalog item to use for 'A4 glossy paper': 'A4 paper' or 'Glossy paper'. We do not have a 
single catalog item that combines size and finish."},{"customer_description":"100 sheets of heavy cardstock 
(white)","match_status":"needs_clarification","item_name":null,"category":null,"units":100,"clarification_question"
:"Please choose the exact catalog item for heavy cardstock: 'Cardstock' or '250 gsm cardstock'. Also confirm color 
(white) if required by the catalog item."},{"customer_description":"100 sheets of colored paper (assorted 
colors)","match_status":"matched","item_name":"Colored 
paper","category":"paper","units":100,"clarification_question":null}],"delivery_by":"2025-04-15","delivery_date_que
stion":null}

[Step 1: Duration 37.03 seconds| Input tokens: 3,009 | Output tokens: 6,442]

{
  "items": [
    {
      "customer_description": "200 sheets of A4 glossy paper",
      "match_status": "needs_clarification",
      "item_name": null,
      "category": null,
      "units": 200,
      "clarification_question": "Please confirm which catalog item to use for 'A4 glossy paper': 'A4 paper' or 'Glossy paper'. We do not have a single catalog item that combines size and finish."
    },
    {
      "customer_description": "100 sheets of heavy cardstock (white)",
      "match_status": "needs_clarification",
      "item_name": null,
      "category": null,
      "units": 100,
      "clarification_question": "Please choose the exact catalog item for heavy cardstock: 'Cardstock' or '250 gsm cardstock'. Also confirm color (white) if required by the catalog item."
    },
    {
      "customer_description": "100 sheets of colored paper (assorted colors)",
      "match_status": "matched",
      "item_name": "Colored paper",
      "category": "paper",
      "units": 100,
      "clari